In [0]:
silverTablName = "saleslake_dev.silver_dev.cleanedInvoice"
goldTablName = "saleslake_dev.gold_dev.dim_invoice"

# Create the target table if it doesn't exist
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {goldTablName} (
    invoice_id BIGINT,
    invoice_number STRING,
    customer_id INT,
    invoice_date DATE,
    due_date DATE,
    subtotal_amount DECIMAL(18,2),
    discount_code STRING,
    discount_amount DECIMAL(18,2),
    tax_amount DECIMAL(18,2),
    total_amount DECIMAL(18,2),
    payment_status STRING,
    payment_method STRING,
    payment_date DATE,
    currency STRING,
    region STRING,
    store_id INT,
    channel STRING,
    created_by STRING,
    customer_sk BIGINT,
    is_active STRING,
    rec_version INT,
    start_effective_ts TIMESTAMP,
    end_effective_ts TIMESTAMP
)
USING DELTA
""")

spark.sql(f"""
MERGE INTO {goldTablName} tgt
USING (
    WITH latest_inv_silver AS (
        SELECT *
        FROM {silverTablName}
        WHERE ingest_ts > (
            SELECT COALESCE(MAX(start_effective_ts), TO_TIMESTAMP('1990-01-01','yyyy-MM-dd'))
            FROM {goldTablName}
        )
    ),
    latest_rm_dup_silver AS (
        SELECT * FROM (
            SELECT *,
                   ROW_NUMBER() OVER (PARTITION BY invoice_id ORDER BY ingest_ts DESC) AS rn
            FROM latest_inv_silver
        ) WHERE rn = 1
    ),
    silver_gold_rec AS (
        SELECT
            s.*,
            g.customer_sk          AS customer_sk,
            g.is_active            AS is_active,
            g.rec_version          AS rec_version,
            g.start_effective_ts   AS start_effective_ts,
            g.end_effective_ts     AS end_effective_ts,
            CASE WHEN g.customer_sk IS NULL THEN 1 ELSE g.rec_version + 1 END AS new_rec_version,
            CASE
                WHEN g.customer_sk IS NULL THEN 'NEW'
                WHEN NOT (s.subtotal_amount  <=> g.subtotal_amount)
                  OR NOT (s.discount_code    <=> g.discount_code)
                  OR NOT (s.discount_amount  <=> g.discount_amount)
                  OR NOT (s.tax_amount       <=> g.tax_amount)
                  OR NOT (s.total_amount     <=> g.total_amount)
                  OR NOT (s.payment_status   <=> g.payment_status)
                  OR NOT (s.payment_method   <=> g.payment_method)
                  OR NOT (s.payment_date     <=> g.payment_date)
                  OR NOT (s.due_date         <=> g.due_date)
                  OR NOT (s.customer_id      <=> g.customer_id)
                  OR NOT (s.region           <=> g.region)
                  OR NOT (s.store_id         <=> g.store_id)
                  OR NOT (s.channel          <=> g.channel)
                THEN 'CHANGE'
                ELSE 'NO_CHANGE'
            END AS rec_flag
        FROM latest_rm_dup_silver s
        LEFT JOIN (SELECT * FROM {goldTablName} WHERE is_active = 'Y') g
               ON s.invoice_id = g.invoice_id
    ),
    insert_flag AS (
        SELECT
            NULL AS inv_merge_key,
            invoice_id, invoice_number, customer_id, invoice_date, due_date,
            subtotal_amount, discount_code, discount_amount, tax_amount, total_amount,
            payment_status, payment_method, payment_date,
            currency, region, store_id, channel, created_by,
            customer_sk, is_active, rec_version, start_effective_ts, end_effective_ts,
            new_rec_version, rec_flag,
            'INSERT' AS merge_flag
        FROM silver_gold_rec
        WHERE rec_flag IN ('NEW','CHANGE')
    ),
    changed_flag AS (
        SELECT
            invoice_id AS inv_merge_key,
            invoice_id, invoice_number, customer_id, invoice_date, due_date,
            subtotal_amount, discount_code, discount_amount, tax_amount, total_amount,
            payment_status, payment_method, payment_date,
            currency, region, store_id, channel, created_by,
            customer_sk, is_active, rec_version, start_effective_ts, end_effective_ts,
            new_rec_version, rec_flag,
            'UPDATE' AS merge_flag
        FROM silver_gold_rec
        WHERE rec_flag = 'CHANGE'
    )
    SELECT * FROM changed_flag
    UNION ALL
    SELECT * FROM insert_flag
) src
ON tgt.invoice_id = src.inv_merge_key

WHEN MATCHED AND src.merge_flag = 'UPDATE' THEN UPDATE SET
    is_active        = 'N',
    end_effective_ts = current_timestamp()

WHEN NOT MATCHED AND src.merge_flag = 'INSERT' THEN INSERT (
    invoice_id, invoice_number, customer_id, invoice_date, due_date,
    subtotal_amount, discount_code, discount_amount, tax_amount, total_amount,
    payment_status, payment_method, payment_date,
    currency, region, store_id, channel, created_by,
    is_active, rec_version, start_effective_ts, end_effective_ts
)
VALUES (
    invoice_id, invoice_number, customer_id, invoice_date, due_date,
    subtotal_amount, discount_code, discount_amount, tax_amount, total_amount,
    payment_status, payment_method, payment_date,
    currency, region, store_id, channel, created_by,
    'Y', new_rec_version, current_timestamp(), TO_TIMESTAMP('9999-12-31','yyyy-MM-dd')
)
""")